# Kaggle runner: train / evaluate on a free GPU  (issue #10)

Runs `scripts/train.py` / `scripts/evaluate.py` from a fresh clone of the repo. All logic lives in the `galaxy`
package; this notebook only clones, installs, sets paths, runs your commands and exports the results.

**One-time setup per notebook**
1. Download this file from GitHub (**Download raw file** icon on the file page), then kaggle.com →
   **Create → New Notebook** → **File → Import Notebook** → drag the `.ipynb` in → **Import**.
2. Right-hand panel → **Session options**: Accelerator **GPU T4 ×2** (or T4 x1), **Internet On**.
3. Right-hand panel → **Input → + Add Input** → search `galaxy10-decals-processed-v1` (*Shared with you* /
   *Your Work*) → **⊕**. See `docs/KAGGLE_DATASET.md`.

Nothing else needs changing except `RUN_CMDS` (cell 4): the install extras and paths are already set.

**Every run**
1. Add a row to `docs/RUNS.md` (status `running`, your Kaggle account).
2. Edit `BRANCH` (cell 1) and `RUN_CMDS` (cell 4). Smoke-test first: `--set train.max_epochs=1 data.fraction=0.1`.
3. Use **Save Version → Save & Run All (Commit)**. It runs in the background (up to 12 h), keeps
   `/kaggle/working` as the version's **Output**, and **waits in the GPU queue by itself** when Kaggle is busy, so
   you can close the tab. Use **Run All** by hand only for quick tests.
4. Cell 6 prints the git steps for committing the results (CONTRIBUTING → *Results workflow*).

**Resuming after the 12 h limit:** open the finished/failed version → **Output** → **New Dataset** (or add this
notebook's output as an input via *Add Input → Your Work*). Set `RESUME_FROM` (cell 3) to the attached folder that
contains `checkpoints/`, add `--resume` to the same commands and run again. Training continues exactly from `last.pt`.

`/kaggle/working` keeps `checkpoints/` (≈ 2 × model size per run) and `results_export/` (+ `.zip`). Its limit is 20 GB.

## 1 · Clone the repo at a branch or commit

In [ ]:
BRANCH = "main"     # branch, tag or commit hash to run
REPO_URL = "https://github.com/ErenKerimoglu/CS464_Project.git"
REPO_DIR = "/tmp/CS464_Project"     # outside /kaggle/working, so the Output only holds checkpoints + results

import os, subprocess, sys, time
SESSION_START = time.time()
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"], check=False)   # no-op for a commit hash
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("running commit", COMMIT)
!git -C {REPO_DIR} log --oneline -1

## 2 · Install the package (torch is preinstalled on Kaggle)

In [ ]:
# "all" (incl. escnn) fails on Kaggle's Python 3.13. Add ",equivariant" only for escnn runs on Python <= 3.12 (#10).
EXTRAS = "deep,classical,tuning"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)  # stops on failure
sys.path.insert(0, REPO_DIR)        # this kernel started before the install
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3 · Paths: attached dataset, checkpoints in `/kaggle/working`

`setup_env()` exports `GALAXY_PROCESSED_DIR` (the attached dataset), `GALAXY_CKPT_DIR=/kaggle/working/checkpoints`
and `GALAXY_DATA_DIR=/kaggle/working/data` for every command started below.

In [ ]:
RESUME_FROM = ""    # e.g. "/kaggle/input/my-previous-run-output" (a folder with checkpoints/ inside); "" = fresh start

import shutil
from galaxy.utils.env import setup_env
paths = setup_env()
if not os.path.exists(os.path.join(paths["processed_dir"], "images_256.npy")):
    raise FileNotFoundError("images_256.npy not found: attach the galaxy10-decals-processed-v1 dataset (Input → Add Input)")
if RESUME_FROM:
    src = os.path.join(RESUME_FROM, "checkpoints")
    shutil.copytree(src, paths["ckpt_dir"], dirs_exist_ok=True)
    print("copied previous checkpoints from", src)

## 4 · Commands to run

One or more commands, run in order from the repo root. A failing command stops the rest.
Check `docs/RUNS.md` first so nobody trains the same run twice.

In [ ]:
RUN_CMDS = [
    "python scripts/train.py --config configs/models/small_cnn.yaml --seed 0 --set train.max_epochs=1 data.fraction=0.1 data.in_memory=true",
    # "python scripts/train.py --config configs/models/resnet50.yaml --seed 0 --set experiment=prelim run_name=resnet50_rotflip data.in_memory=true",
    # 'python scripts/run_rotation_sweep.py --runs "results/runs/prelim/resnet50_*/seed0" --name prelim_rotation --figure results/figures/e1/prelim_accuracy_vs_angle',
    # "python scripts/evaluate.py --run-dir results/runs/model_family/resnet50/seed0 --eval test --tta8",
]

## 5 · Run

In [ ]:
from galaxy.utils.env import run_commands
run_commands(RUN_CMDS, cwd=REPO_DIR)

## 6 · Export results → `/kaggle/working/results_export.zip` and print the git steps

Exports the new files of `results/runs`, `results/tables` and `results/figures` (e.g. the rotation sweep).

In [ ]:
from galaxy.utils.env import export_results, results_instructions
EXPORT_DIR = "/kaggle/working/results_export"
run_dirs = export_results(REPO_DIR, EXPORT_DIR, newer_than=SESSION_START)
print(results_instructions(run_dirs, EXPORT_DIR + ".zip", COMMIT))
print()
!du -sh /kaggle/working/checkpoints /kaggle/working/results_export* 2>/dev/null